# Data Readiness and Phone Search

Run cells in order with a Python 3 kernel. All prose and summaries are English. Source-language product data remains in the database. This notebook reads generated artifacts and does not rebuild or change source data.

The saved outputs were executed sequentially with Python by the builder. A Jupyter kernel launch was not tested in this environment.

In [1]:
from pathlib import Path
import sys, json, sqlite3
from contextlib import closing
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/search_catalog.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.search_catalog import search
DB = ROOT / "data/processed/catalog.sqlite"
assert DB.exists(), "Run python scripts/prepare_data.py first"
print("Project and database located.")

Project and database located.


## Source Counts

Check the live database against the recorded baseline.

In [2]:
profile = json.loads((ROOT / "reports/data_profile.json").read_text())
with closing(sqlite3.connect(DB.as_uri() + "?mode=ro", uri=True)) as db:
    for table in ("shops", "products", "offers"):
        count = db.execute("SELECT COUNT(*) FROM " + table).fetchone()[0]
        assert count == profile["tables"][table]["rows"]
        print(f"{table}: {count:,}")

shops: 6,476
products: 225,903
offers: 331,506


## Category Selection

Phones are the first demo category because 336 of 814 products have multiple screened sellers. Counts do not establish variant equivalence or current availability.

In [3]:
quality = json.loads((ROOT / "reports/category_quality.json").read_text())
for row in quality["categories"]:
    if row["label"] != row["category_id"]:
        print(f"{row['label']}: {row['products']} products, {row['comparable_products']} comparable, {row['shops']} shops")
print("Price screening:")
print(json.dumps(quality["price_status_counts"], indent=2))

Laptops: 505 products, 51 comparable, 24 shops
Speakers and audio systems: 1063 products, 267 comparable, 213 shops
Power banks: 755 products, 199 comparable, 166 shops
Mobile phones: 814 products, 336 comparable, 127 shops
Headphones and headsets: 1339 products, 336 comparable, 244 shops
Price screening:
{
  "category_outlier": 3837,
  "insufficient_reference": 22948,
  "missing": 15083,
  "screened": 289638
}


## Try a Search

Edit the query, budget, location, and radius below. Distance is straight-line distance, not route distance. Non-capacity query tokens must occur in both product metadata and seller title. English and Persian GB/TB aliases are normalized. Explicit storage queries require matching capacity evidence in both titles; conflicting or missing evidence is excluded. Bare numbers remain lexical terms.

Results are offers, not merged variants. Price is recorded and unverified. Inventory remains unknown. A used-condition hint is based on title mentions; no mention means unknown, not new.

In [4]:
QUERY = "A55"
BUDGET_TOMAN = 100_000_000
LATITUDE, LONGITUDE = 35.7, 51.4
RADIUS_KM = 10
response = search(DB, QUERY, BUDGET_TOMAN, LATITUDE, LONGITUDE, RADIUS_KM)
print(f"Matching offers: {response['total_matches']}")
for row in response["results"]:
    print(json.dumps({k: row[k] for k in ("product_id", "offer_id", "shop_id", "price_toman", "distance_km", "inventory_status", "variant_status")}, ensure_ascii=True))

Matching offers: 3
{"product_id": "product_b16e428b704b8ea30f4e9445", "offer_id": "offer_65b971d93d221620b40bbf27", "shop_id": "shop_b6c2ac1ce44ee8f6be5cc7c2", "price_toman": 85000000, "distance_km": 1.404105254021062, "inventory_status": "unknown", "variant_status": "requires_review"}
{"product_id": "product_b16e428b704b8ea30f4e9445", "offer_id": "offer_78e97a00642f3c874a05d9fb", "shop_id": "shop_0b2d387c5a2ecd7d8d3e5065", "price_toman": 95000000, "distance_km": 1.1997590407262746, "inventory_status": "unknown", "variant_status": "requires_review"}
{"product_id": "product_a9e759a72c31e1dea46f17e5", "offer_id": "offer_ab9d4ceb8477ac1ac8764376", "shop_id": "shop_398fa7e4123f7a7abb89c606", "price_toman": 98000000, "distance_km": 1.141382602048157, "inventory_status": "unknown", "variant_status": "requires_review"}


## Constraint Evaluation

Twenty fixed queries cover models, storage terms, budgets, radius changes, missing models, and letter case. These are automated constraint checks. Relevance labels remain pending; an empty result does not prove good retrieval. Fixture tests separately cover missing prices, suspect offers, missing coordinates, and exact radius boundaries.

In [5]:
evaluation = json.loads((ROOT / "reports/search_evaluation.json").read_text())
for row in evaluation["cases"]:
    print(f"{row['case']}: matches={row['total_matches']}, constraints_pass={row['constraints_pass']}")
assert all(row["constraints_pass"] for row in evaluation["cases"])
print("Manual relevance review remains pending.")

model_a55: matches=7, constraints_pass=True
model_a35: matches=2, constraints_pass=True
model_a15: matches=0, constraints_pass=True
model_s23: matches=6, constraints_pass=True
model_s24: matches=12, constraints_pass=True
model_iphone: matches=66, constraints_pass=True
model_redmi: matches=209, constraints_pass=True
model_poco: matches=94, constraints_pass=True
storage_128: matches=1, constraints_pass=True
storage_256: matches=6, constraints_pass=True
storage_512: matches=1, constraints_pass=True
storage_unit: matches=6, constraints_pass=True
budget_low: matches=0, constraints_pass=True
budget_mid: matches=3, constraints_pass=True
radius_zero: matches=0, constraints_pass=True
radius_small: matches=0, constraints_pass=True
radius_large: matches=7, constraints_pass=True
unknown_model: matches=0, constraints_pass=True
near_model: matches=32, constraints_pass=True
case_normalization: matches=7, constraints_pass=True
Manual relevance review remains pending.


## Reviewed Storage Example

Seven A55 offers were inspected by the assistant: six mention 256 GB and one mentions 128 GB. Two of the six mention used/stock condition. This is a development sample, not an independent human evaluation or held-out accuracy benchmark.

In [6]:
reviewed = search(DB, "A55 256GB", 200_000_000, LATITUDE, LONGITUDE, RADIUS_KM)
for row in reviewed["results"]:
    print(json.dumps({k: row[k] for k in ("offer_id", "storage_gb_evidence", "condition_hint")}, ensure_ascii=True))
print(json.dumps(evaluation["reviewed_sample"], indent=2))

{"offer_id": "offer_65b971d93d221620b40bbf27", "storage_gb_evidence": [256], "condition_hint": "used_or_refurbished_mentioned"}
{"offer_id": "offer_78e97a00642f3c874a05d9fb", "storage_gb_evidence": [256], "condition_hint": "used_or_refurbished_mentioned"}
{"offer_id": "offer_81116b69cd21638a0dd7a2c9", "storage_gb_evidence": [256], "condition_hint": "unknown"}
{"offer_id": "offer_49e13cd42dc4ab68414c24ea", "storage_gb_evidence": [256], "condition_hint": "unknown"}
{"offer_id": "offer_2d1c98f49ed3a847d8d81a10", "storage_gb_evidence": [256], "condition_hint": "unknown"}
{"offer_id": "offer_48ca39a15d74ce7b5507af32", "storage_gb_evidence": [256], "condition_hint": "unknown"}
{
  "reviewer": "assistant_source_title_review",
  "positive_examples_retrieved": 6,
  "negative_examples_retrieved": 0,
  "scope": "Development regression on seven inspected offers, not held-out retrieval accuracy"
}


## Next Checks

Review broader model families and ambiguous titles. Independent relevance judgments remain pending. Exact RAM, color, warranty, and condition equivalence are not established. The baseline uses local SQLite and bounded category scans; it is not a production search service.